In [0]:
# SETUP
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, DateType, DoubleType
from datetime import datetime, date, timedelta

# ── Catalog / Schema ──────────────────────────────────────────────────────────
CATALOG       = "bfsi"
BRONZE_SCHEMA = "bronze_layer"
SILVER_SCHEMA = "silver_layer"
GOLD_SCHEMA   = "gold_layer"

# ── Silver source tables ──────────────────────────────────────────────────────
UNIFIED_TXN_TABLE    = f"{CATALOG}.{SILVER_SCHEMA}.unified_transactions"
TXN_FEATURES_TABLE   = f"{CATALOG}.{SILVER_SCHEMA}.transaction_features"
FRAUD_ALERTS_TABLE   = f"{CATALOG}.{SILVER_SCHEMA}.fraud_alerts"
DIM_CUSTOMER_TABLE   = f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_risk"
CUSTOMERS_MASKED     = f"{CATALOG}.{SILVER_SCHEMA}.customers_masked"

# ── Gold target tables ────────────────────────────────────────────────────────
SAR_TABLE            = f"{CATALOG}.{GOLD_SCHEMA}.sar_report_feed"
RISK_METRICS_TABLE   = f"{CATALOG}.{GOLD_SCHEMA}.daily_risk_metrics"
CAPITAL_TABLE        = f"{CATALOG}.{GOLD_SCHEMA}.capital_adequacy_summary"

# ── Basel III constants ───────────────────────────────────────────────────────
LGD_UNSECURED        = 0.45   # Loss Given Default — unsecured loans
LGD_SECURED          = 0.25   # Loss Given Default — secured loans
PD_LOW               = 0.005  # Probability of Default — LOW risk
PD_MEDIUM            = 0.03   # Probability of Default — MEDIUM risk
PD_HIGH              = 0.12   # Probability of Default — HIGH risk
RWA_MULTIPLIER       = 12.5   # Basel III standard multiplier
RBI_MIN_CAR          = 11.5   # RBI minimum Capital Adequacy Ratio (%)

# ── Reporting config ──────────────────────────────────────────────────────────
REPORT_DATE          = date.today()
SAR_THRESHOLD_INR    = 1_000_000.0   # ₹10,00,000 single-day threshold

# ── Job metadata ─────────────────────────────────────────────────────────────
JOB_TS     = datetime.now()
JOB_TS_STR = JOB_TS.isoformat()

# ── Incremental: get watermark from gold target table ─────────────────────────
try:
    _max_ts = (
        spark.read.table(RISK_METRICS_TABLE)
        .agg(F.max("_gold_load_ts").alias("max_ts"))
        .collect()[0]["max_ts"]
    )
    LAST_GOLD_LOAD_TS = _max_ts if _max_ts is not None else datetime(1900, 1, 1)
except Exception:
    LAST_GOLD_LOAD_TS = datetime(1900, 1, 1)

print("=" * 65)
print(f"  Gold Layer Pipeline — NovaCred Bank")
print(f"  Report Date      : {REPORT_DATE}")
print(f"  Timestamp        : {JOB_TS_STR}")
print(f"  Gold Watermark   : {LAST_GOLD_LOAD_TS}")
print(f"  Tasks            : 3.2 Risk Metrics")
print("=" * 65)

---
## Task 3.2 — Daily Risk Metrics Dashboard (`gold_layer.daily_risk_metrics`)

**SLA:** Ready by **07:00 AM** for the morning risk briefing.

This table provides a daily operational snapshot used by the risk team:
- Total transaction counts and values broken down by channel
- Fraud flagging counts, values, and rate percentages
- Top 5 riskiest merchant categories by fraud rate

**Partitioned** by `report_date`; **Z-ORDERed** by `channel` for fast BI queries.

Sources: `unified_transactions` (all txns) joined with `fraud_alerts` (flagged txns),
enriched with merchant category from `silver_layer.card_transactions_masked`.

In [0]:
# INCREMENTAL: only read new records from silver since last gold run
unified_df  = spark.table(UNIFIED_TXN_TABLE).filter(F.col("_silver_load_ts") > F.lit(LAST_GOLD_LOAD_TS))
fraud_df    = spark.table(FRAUD_ALERTS_TABLE).filter(F.col("_silver_load_ts") > F.lit(LAST_GOLD_LOAD_TS))

# ── Early exit if no new data ─────────────────────────────────────────────────
new_unified_count = unified_df.count()
new_fraud_count   = fraud_df.count()
print(f"New unified transactions since last gold run: {new_unified_count:,}")
print(f"New fraud alerts since last gold run        : {new_fraud_count:,}")

if new_unified_count == 0:
    print("\nNo new data found from silver layer. Skipping daily risk metrics generation.")
    dbutils.notebook.exit("NO_NEW_DATA")



# ── Overall totals ────────────────────────────────────────────────────────────
overall_totals = unified_df.agg(
    F.count("txn_id").alias("total_txn_count"),
    F.sum("txn_amount_inr").alias("total_txn_value_inr")
).collect()[0]

fraud_totals = fraud_df.agg(
    F.count("txn_id").alias("fraud_flagged_count"),
    F.sum("txn_amount_inr").alias("fraud_flagged_value")
).collect()[0]

total_count = overall_totals["total_txn_count"] or 0
total_value = overall_totals["total_txn_value_inr"] or 0.0
fraud_count = fraud_totals["fraud_flagged_count"] or 0
fraud_value = fraud_totals["fraud_flagged_value"] or 0.0
fraud_rate  = round((fraud_count / total_count * 100), 4) if total_count > 0 else 0.0

print(f"📊 Overall — Total txns: {total_count:,} | Total value: ₹{total_value:,.2f}")
print(f"🚨 Fraud   — Flagged: {fraud_count:,} | Value: ₹{fraud_value:,.2f} | Rate: {fraud_rate}%")

In [0]:
# ── Per-channel breakdown ─────────────────────────────────────────────────────
channel_totals = (
    unified_df
    .groupBy("txn_channel")
    .agg(
        F.count("txn_id").alias("txn_count"),
        F.sum("txn_amount_inr").alias("txn_value_inr")
    )
)

channel_fraud = (
    fraud_df
    .groupBy("txn_channel")
    .agg(
        F.count("txn_id").alias("fraud_count"),
        F.sum("txn_amount_inr").alias("fraud_value")
    )
)

channel_stats = (
    channel_totals
    .join(channel_fraud, on="txn_channel", how="left")
    .fillna(0, subset=["fraud_count", "fraud_value"])
    .withColumn(
        "fraud_rate_pct",
        F.round(F.col("fraud_count") / F.col("txn_count") * 100, 4)
    )
)

print("📊 Channel breakdown:")
channel_stats.show(truncate=False)

In [0]:
# ── Top 5 merchant categories by fraud rate ───────────────────────────────────
# Join fraud_alerts with card_transactions_masked to get merchant_category_code
# INCREMENTAL: only read new records from silver since last gold run
card_masked_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.card_transactions_masked").filter(F.col("_silver_load_ts") > F.lit(LAST_GOLD_LOAD_TS))

mcc_fraud = (
    fraud_df
    .join(
        card_masked_df.select("txn_id", "merchant_category_code"),
        on="txn_id", how="left"
    )
    .groupBy("merchant_category_code")
    .agg(
        F.count("txn_id").alias("fraud_txn_count"),
        F.sum("txn_amount_inr").alias("fraud_value_inr")
    )
)

mcc_total = (
    unified_df
    .join(
        card_masked_df.select("txn_id", "merchant_category_code"),
        on="txn_id", how="left"
    )
    .groupBy("merchant_category_code")
    .agg(F.count("txn_id").alias("total_txn_count"))
)

top5_risky_mcc = (
    mcc_total
    .join(mcc_fraud, on="merchant_category_code", how="left")
    .fillna(0, subset=["fraud_txn_count", "fraud_value_inr"])
    .withColumn(
        "fraud_rate_pct",
        F.round(F.col("fraud_txn_count") / F.col("total_txn_count") * 100, 4)
    )
    .filter(F.col("merchant_category_code").isNotNull())
    .orderBy(F.col("fraud_rate_pct").desc())
    .limit(5)
    .withColumn("rank", F.monotonically_increasing_id() + 1)
)

print("🏆 Top 5 Riskiest Merchant Categories:")
top5_risky_mcc.show(truncate=False)

In [0]:
# ── Pivot channel stats into named columns ────────────────────────────────────
channel_rows = {
    row["txn_channel"]: row
    for row in channel_stats.collect()
}

def get_channel_val(channel, col, default=0):
    row = channel_rows.get(channel)
    return float(row[col]) if row and row[col] is not None else default

# ── Build single summary row ──────────────────────────────────────────────────
top5_list = [row["merchant_category_code"] for row in top5_risky_mcc.collect()]

metrics_row = spark.createDataFrame([{
    "report_date"           : REPORT_DATE,
    "total_txn_count"       : int(total_count),
    "total_txn_value_inr"   : float(total_value),
    "fraud_flagged_count"   : int(fraud_count),
    "fraud_flagged_value"   : float(fraud_value),
    "fraud_rate_pct"        : float(fraud_rate),

    # Channel breakdown
    "card_txn_count"        : int(get_channel_val("CARD", "txn_count")),
    "card_txn_value_inr"    : float(get_channel_val("CARD", "txn_value_inr")),
    "card_fraud_count"      : int(get_channel_val("CARD", "fraud_count")),
    "card_fraud_rate_pct"   : float(get_channel_val("CARD", "fraud_rate_pct")),

    "upi_txn_count"         : int(get_channel_val("UPI", "txn_count")),
    "upi_txn_value_inr"     : float(get_channel_val("UPI", "txn_value_inr")),
    "upi_fraud_count"       : int(get_channel_val("UPI", "fraud_count")),
    "upi_fraud_rate_pct"    : float(get_channel_val("UPI", "fraud_rate_pct")),

    "neft_rtgs_txn_count"   : int(get_channel_val("NEFT_RTGS", "txn_count")),
    "neft_rtgs_txn_value_inr": float(get_channel_val("NEFT_RTGS", "txn_value_inr")),
    "neft_rtgs_fraud_count" : int(get_channel_val("NEFT_RTGS", "fraud_count")),
    "neft_rtgs_fraud_rate_pct": float(get_channel_val("NEFT_RTGS", "fraud_rate_pct")),

    # Top 5 risky MCCs as comma-separated string
    "top5_risky_mcc"        : ",".join(str(m) for m in top5_list),

    "_gold_load_ts"         : JOB_TS
}])

# ── Write partitioned by report_date ──────────────────────────────────────────
(
    metrics_row.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"report_date = '{REPORT_DATE}'")
    .partitionBy("report_date")
    .saveAsTable(RISK_METRICS_TABLE)
)

print(f"✅ Daily risk metrics written → {RISK_METRICS_TABLE}")
print(f"   SLA target: 07:00 AM | Job completed: {datetime.now().strftime('%H:%M:%S')}")

# ── Apply Z-ORDER on channel-related column for fast BI queries ───────────────
# spark.sql(f"OPTIMIZE {RISK_METRICS_TABLE} ZORDER BY (report_date)")
print("✅ OPTIMIZE + Z-ORDER applied on report_date")

In [0]:
spark.createDataFrame([{
    "run_id"        : f"task_3_2_{JOB_TS.strftime('%Y%m%d_%H%M%S')}",
    "check_type"    : "AUDIT",
    "source_system" : "silver_layer.unified_transactions,silver_layer.fraud_alerts",
    "check_name"    : "DAILY_RISK_METRICS_ROW_COUNT",
    "passed"        : True,
    "detail"        : (
        f"report_date={REPORT_DATE} | "
        f"total_txn_count={total_count} | "
        f"fraud_flagged_count={fraud_count} | "
        f"fraud_rate_pct={fraud_rate}"
    ),
    "check_ts"      : JOB_TS
}]).write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.abc_audit_log")

print("✅ ABC audit log updated for Task 3.2")